# 실습과제 Q4: 모델평가와 교차검증

**관련 차시**: 17차시 - 모델평가와 반복검증  
**난이도**: ★★★☆☆ (중급)

---

## 학습 목표

1. 혼동행렬(Confusion Matrix)을 만들고 해석하기
2. 정밀도(Precision), 재현율(Recall), F1 스코어 계산하기
3. K-Fold 교차검증으로 모델의 일반화 성능 평가하기
4. 과적합 여부를 판단하고 하이퍼파라미터 조정하기

In [ ]:
# 필요한 라이브러리 임포트
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (confusion_matrix, classification_report,
                             precision_score, recall_score, f1_score,
                             ConfusionMatrixDisplay)

In [ ]:
# 데이터 로드 및 준비 (Q3에서 이어서)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv"
df = pd.read_csv(url)

feature_cols = ['Air temperature [K]', 'Process temperature [K]',
                'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
X = df[feature_cols]
y = df['Machine failure']

# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 모델 학습
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("데이터 준비 및 모델 학습 완료!")

---

## 문제 1: 혼동행렬 작성 및 시각화 (20점)

모델의 예측 결과로 **혼동행렬(Confusion Matrix)**을 만들고 시각화하세요.

In [ ]:
# TODO: 혼동행렬 계산 (빈칸 채우기)
cm = confusion_matrix(______, ______)  # y_test, y_pred 순서로
print("혼동행렬:")
print(cm)

In [ ]:
# 시각화
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Normal', 'Failure'])
disp.plot(ax=ax, cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# 혼동행렬의 각 요소 분리
TN, FP, FN, TP = cm.ravel()
print(f"TN (True Negative): {TN}")
print(f"FP (False Positive): {FP}")
print(f"FN (False Negative): {FN}")
print(f"TP (True Positive): {TP}")

### 문제 1 답변 작성

| 지표 | 값 | 설명 |
|------|:--:|------|
| TN | ______ | 정상을 정상으로 예측 |
| FP | ______ | 정상을 불량으로 예측 |
| FN | ______ | 불량을 정상으로 예측 (미탐) |
| TP | ______ | 불량을 불량으로 예측 |

---

## 문제 2: 혼동행렬 해석 (제조 관점) (20점)

제조 현장 관점에서 각 지표의 의미를 해석하세요.

### 문제 2 답변 작성

**제조 현장에서의 의미:**

- **TN**: 정상 제품을 정상으로 판정 → ______________________
- **FP**: 정상 제품을 불량으로 판정 → ______________________
- **FN**: 불량 제품을 정상으로 판정 → ______________________
- **TP**: 불량 제품을 불량으로 판정 → ______________________

**추가 질문:**

- FN(미탐)이 많으면 어떤 문제가 발생하나요?: ______________________
- FP(오탐)가 많으면 어떤 비용이 발생하나요?: ______________________
- 제조 품질 관리에서는 FN과 FP 중 어떤 것을 더 줄여야 하나요?: ______________________

---

## 문제 3: 정밀도, 재현율, F1 스코어 계산 (25점)

세 가지 평가 지표를 계산하고 해석하세요.

In [ ]:
# TODO: 개별 지표 계산 (빈칸 채우기)
precision = precision_score(y_test, y_pred)
recall = ______(y_test, y_pred)  # recall_score 함수 사용
f1 = f1_score(y_test, y_pred)

print(f"정밀도 (Precision): {precision:.4f}")
print(f"재현율 (Recall): {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

In [ ]:
# 수식으로 직접 계산하여 확인
precision_manual = TP / (TP + FP) if (TP + FP) > 0 else 0
recall_manual = TP / (TP + FN) if (TP + FN) > 0 else 0
f1_manual = 2 * (precision_manual * recall_manual) / (precision_manual + recall_manual) if (precision_manual + recall_manual) > 0 else 0

print(f"\n수식으로 계산:")
print(f"정밀도 = TP/(TP+FP) = {TP}/({TP}+{FP}) = {precision_manual:.4f}")
print(f"재현율 = TP/(TP+FN) = {TP}/({TP}+{FN}) = {recall_manual:.4f}")
print(f"F1 = 2×(P×R)/(P+R) = {f1_manual:.4f}")

In [ ]:
# 전체 분류 리포트
print("\n분류 리포트:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Failure']))

### 문제 3 답변 작성

- 정밀도: ______
- 재현율: ______
- F1 Score: ______

- 재현율이 중요한 이유 (제조 관점): ______________________
- 현재 모델의 재현율로 실무에서 사용 가능한가?: ______________________

---

## 문제 4: K-Fold 교차검증 (20점)

**5-Fold 교차검증**을 수행하여 모델의 일반화 성능을 평가하세요.

In [ ]:
# TODO: 5-Fold 교차검증 (빈칸 채우기)
cv_scores = cross_val_score(model, X, y, cv=______, scoring='accuracy')  # cv=5

print(f"각 Fold의 정확도: {cv_scores}")
print(f"평균 정확도: {cv_scores.mean():.4f}")
print(f"표준편차: {cv_scores.std():.4f}")

### 문제 4 답변 작성

- Fold 1 정확도: ______
- Fold 2 정확도: ______
- Fold 3 정확도: ______
- Fold 4 정확도: ______
- Fold 5 정확도: ______
- 평균 정확도: ______
- 표준편차: ______

- Fold 간 정확도 편차가 큰가/작은가?: ______________________
- 교차검증의 장점: ______________________

---

## 문제 5: 과적합 진단 및 max_depth 조정 (15점)

max_depth를 변경하면서 과적합 여부를 진단하세요.

In [ ]:
depths = [2, 3, 4, 5, 7, 10, None]
train_scores = []
test_scores = []
cv_means = []

for depth in depths:
    model_exp = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model_exp.fit(X_train, y_train)
    
    train_acc = model_exp.score(X_train, y_train)
    test_acc = model_exp.score(X_test, y_test)
    cv_score = cross_val_score(model_exp, X, y, cv=5).mean()
    
    train_scores.append(train_acc)
    test_scores.append(test_acc)
    cv_means.append(cv_score)
    
    depth_str = str(depth) if depth else 'None'
    print(f"max_depth={depth_str:>4}: 학습={train_acc:.4f}, 테스트={test_acc:.4f}, CV={cv_score:.4f}")

In [ ]:
# 시각화
plt.figure(figsize=(10, 6))
x_labels = [str(d) if d else 'None' for d in depths]
plt.plot(x_labels, train_scores, 'o-', label='Train')
plt.plot(x_labels, test_scores, 's-', label='Test')
plt.plot(x_labels, cv_means, '^-', label='CV Mean')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Overfitting Diagnosis: Train vs Test vs CV')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 문제 5 답변 작성

- max_depth 증가 시 학습 정확도 변화: ______________________
- max_depth 증가 시 테스트/CV 정확도 변화: ______________________
- 과적합이 시작되는 max_depth: ______
- 최적의 max_depth: ______

---

## 심화 문제 (선택)

### 심화 1: 재현율 기준 교차검증

In [ ]:
cv_recall = cross_val_score(model, X, y, cv=5, scoring='recall')
print(f"재현율 기반 CV: {cv_recall.mean():.4f} (+/- {cv_recall.std():.4f})")

### 심화 2: 클래스 가중치 적용

In [ ]:
model_weighted = DecisionTreeClassifier(max_depth=3, random_state=42,
                                         class_weight='balanced')
model_weighted.fit(X_train, y_train)
y_pred_weighted = model_weighted.predict(X_test)
print("클래스 가중치 적용 후:")
print(classification_report(y_test, y_pred_weighted, target_names=['Normal', 'Failure']))

---

## 과제 완료 체크리스트

- [ ] 문제 1: 혼동행렬 작성 및 TN/FP/FN/TP 식별
- [ ] 문제 2: 제조 관점 해석 및 FN/FP 중요성 설명
- [ ] 문제 3: 정밀도/재현율/F1 계산 및 해석
- [ ] 문제 4: 5-Fold CV 수행 및 결과 분석
- [ ] 문제 5: 과적합 진단 및 최적 depth 제안
- [ ] 모든 답변 작성 완료